# 03 — Clinical reality check & self-evaluation

Honest reading of what this project is good for — and what it is not.

**Framing:** this is **seizure-marker-proximity detection** on technician workflow annotations (BDSP Neurotech; Morgan et al., 2026), **not** multi-expert clinically validated seizure-interval detection. Author: **Kush Patel**.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display, Markdown

ROOT = Path("..").resolve()
METRICS = ROOT / "docs" / "showcase" / "metrics"
FIGS = ROOT / "docs" / "showcase" / "figures"

headline = json.loads((METRICS / "headline.json").read_text())
test = headline["test_ft_max15"]
forecast = json.loads((METRICS / "forecast_v2_summary.json").read_text())
smooth = json.loads((METRICS / "conformer_ft_test_smooth15.json").read_text())

## Detection looks good on ranking metrics

On the held-out **test** patients (smoothed):

- PR-AUC ≈ **0.46** (~12× prevalence)
- ROC-AUC ≈ **0.86**
- Event-level sensitivity ≈ **60%** with ~**45** false alarms / 24h (SzCORE-style merge)

Those numbers mean the model **ranks** seizure-like windows much better than chance.

In [ ]:
display(Image(filename=str(FIGS / "confusion_matrix.png")))
print(
    f"At F1-ish threshold {smooth['threshold']:.3f}: "
    f"precision={smooth['precision']:.3f}, recall={smooth['sensitivity']:.3f}, "
    f"FA/hour={smooth['false_alarms_per_hour']:.2f}"
)

## Where clinical use would break today

| Target | Precision | False alarms / 24h |
|---|---:|---:|
| Catch 50% of positives (R50) | ~30% | ~371 |
| Catch 70% (R70) | ~15% | ~1,265 |
| Catch 90% (R90) | ~7% | ~5,449 |

A bedside alarm that fires hundreds–thousands of times per day is not deployable without heavy human review, patient-specific tuning, and cleaner labels.

**Also:** labels are technician *point* markers (±30 s), not expert onset/offset. That caps how "clinical-grade" any score can be.

In [ ]:
display(Image(filename=str(FIGS / "operating_points.png")))

## Forecasting (preictal) — separate, mostly negative result

We also tried seizure **prediction** (SOP/SPH preictal windows). Even with cleaner labeling, test lift stayed near chance (~1.1×).
That is an important scientific outcome: **detection ≠ forecasting** on this weak-label scalp setup.

In [ ]:
print("Forecast v2 summary:")
for k, v in forecast.items():
    print(f"  {k}: {v}")

## Self-evaluation

### For an entry-level data scientist breaking into industry

**Grade: strong portfolio piece (A− / A).**

Why it helps you get interviews:
- End-to-end ownership: ingest → labels → cache → train → rigorous eval
- Correct ML hygiene: patient-level splits, imbalance-aware metrics, bootstrap CIs
- Engineering maturity: resumable S3 pipeline, 244GB memmap cache, CLI, tests
- Intellectual honesty: documents failure modes (FA/24h, weak labels, forecast null)

What hiring managers still poke at:
- Not a shipped product / no online serving demo
- Weak labels vs curated clinical datasets (TUH, CHB-MIT with different tradeoffs)
- Large compute story (EC2 + archive) — explain cost/decisions clearly

### For clinical settings

**Grade: research prototype only (not deployment-ready).**

Useful as:
- Offline triage / prioritization research on historical EEG
- A baseline for paper-style benchmarking with PRG + event FA rates

Not ready for:
- Unattended alarms
- Diagnostic claims
- Regulatory pathways without expert-adjudicated labels, prospective study, and human factors work

### Bottom line

This project is **excellent as applied ML + data engineering storytelling**, and **honestly modest as a clinical tool**. That combination is exactly what good scientific portfolios look like.